In [1]:
import os

os.environ["OPENAI_API_KEY"] = "sk-**********************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,  # 稳定输出
    timeout=1200,  # 超时保护（秒）
    max_retries=2,  # 简单重试
)

In [3]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """将两个整数相乘。"""
    return a * b


@tool
def add(a: int, b: int) -> int:
    """将两个整数相加。"""
    return a + b


# 创建工具列表
tools = [multiply, add]

In [4]:
llm_with_tools = llm.bind_tools(tools)

In [6]:
# 提一个需要使用工具的问题
query = "3乘以12等于多少？另外，11加上49等于多少？"
response = llm_with_tools.invoke(query)

print(response)
# 查看模型生成的工具调用
print(response.tool_calls)
# 输出:
# [{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': '...'},
#  {'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': '...'}]

content='' additional_kwargs={'tool_calls': [{'id': 'call_7e2726fe05c3412fbe7169', 'function': {'arguments': '{"a": 3, "b": 12}', 'name': 'multiply'}, 'type': 'function', 'index': 0}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 248, 'total_tokens': 300, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-turbo', 'system_fingerprint': None, 'id': 'chatcmpl-a04a0d3e-0b52-9a93-97d2-e3c42cfb694e', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='run--8acbdc72-1cc0-4d05-85c8-5d17e00f2759-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_7e2726fe05c3412fbe7169', 'type': 'tool_call'}] usage_metadata={'input_tokens': 248, 'output_tokens': 52, 'total_tokens': 300, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}
[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_7e2726fe05c3412fbe7169', 'type